# Intelligent Plant Disease Detection System (IPDDS)
## Targeted Model Training: Potato & Tomato Leaf Diseases
This notebook trains the same CNN as `model_training.ipynb`, using only the Potato and Tomato classes from the KaggleHub PlantVillage training dataset. The selected images are split into training and validation sets.

In [1]:
import os
import sys
import json
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

sys.path.insert(0, os.path.abspath(".."))
from train import DISEASE_METADATA

print(f"TensorFlow Version: {tf.__version__}")

TensorFlow Version: 2.21.0


### 1. Dataset Path & Target Classes
The source directory contains Potato, Tomato, and Pepper categories. Only class folders whose names begin with `Potato` or `Tomato` are selected.

In [2]:
train_path = r"C:\Users\Muneer Hameed\.cache\kagglehub\datasets\moazeldsokyx\plantvillage\versions\1\dataset\train"

if not os.path.isdir(train_path):
    raise FileNotFoundError(f"PlantVillage training directory was not found: {train_path}")

POTATO_TOMATO_CLASSES = sorted(
    folder_name
    for folder_name in os.listdir(train_path)
    if os.path.isdir(os.path.join(train_path, folder_name))
    and folder_name.startswith(("Potato", "Tomato"))
)

if not POTATO_TOMATO_CLASSES:
    raise ValueError(f"No Potato or Tomato class folders were found in: {train_path}")

print(f"Dataset directory: {train_path}")
print(f"Selected classes ({len(POTATO_TOMATO_CLASSES)}):")
for class_name in POTATO_TOMATO_CLASSES:
    print(f"  {class_name}")

Dataset directory: C:\Users\Muneer Hameed\.cache\kagglehub\datasets\moazeldsokyx\plantvillage\versions\1\dataset\train
Selected classes (13):
  Potato___Early_blight
  Potato___Late_blight
  Potato___healthy
  Tomato_Bacterial_spot
  Tomato_Early_blight
  Tomato_Late_blight
  Tomato_Leaf_Mold
  Tomato_Septoria_leaf_spot
  Tomato_Spider_mites_Two_spotted_spider_mite
  Tomato__Target_Spot
  Tomato__Tomato_YellowLeaf__Curl_Virus
  Tomato__Tomato_mosaic_virus
  Tomato_healthy


### 2. Loading Filtered Image Datasets
Each class folder is split separately into 80% training and 20% validation data, ensuring Potato and Tomato images from every disease class appear in both splits.

In [3]:
IMG_SIZE = (128, 128)
BATCH_SIZE = 32
VALIDATION_SPLIT = 0.2
RANDOM_SEED = 42

AUTOTUNE = tf.data.AUTOTUNE
SUPPORTED_IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".gif")
train_examples = []
val_examples = []
rng = np.random.default_rng(RANDOM_SEED)

for class_index, class_name in enumerate(POTATO_TOMATO_CLASSES):
    class_directory = os.path.join(train_path, class_name)
    image_paths = [
        os.path.join(class_directory, file_name)
        for file_name in os.listdir(class_directory)
        if file_name.lower().endswith(SUPPORTED_IMAGE_EXTENSIONS)
    ]
    if len(image_paths) < 2:
        raise ValueError(f"Class {class_name} needs at least two images for a train/validation split.")
    image_paths = np.asarray(sorted(image_paths), dtype=str)
    rng.shuffle(image_paths)
    validation_count = max(1, int(round(len(image_paths) * VALIDATION_SPLIT)))
    val_examples.extend((path, class_index) for path in image_paths[:validation_count])
    train_examples.extend((path, class_index) for path in image_paths[validation_count:])

def create_dataset(examples, shuffle):
    image_paths, labels = zip(*examples)
    dataset = tf.data.Dataset.from_tensor_slices((list(image_paths), list(labels)))
    if shuffle:
        dataset = dataset.shuffle(len(image_paths), seed=RANDOM_SEED, reshuffle_each_iteration=True)

    def load_image(path, label):
        image = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
        image.set_shape((None, None, 3))
        image = tf.image.resize(image, IMG_SIZE)
        return image, label

    return dataset.map(load_image, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = create_dataset(train_examples, shuffle=True)
val_ds = create_dataset(val_examples, shuffle=False)

print(f"Classes loaded ({len(POTATO_TOMATO_CLASSES)} total): {POTATO_TOMATO_CLASSES}")

Classes loaded (13 total): ['Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Tomato_Bacterial_spot', 'Tomato_Early_blight', 'Tomato_Late_blight', 'Tomato_Leaf_Mold', 'Tomato_Septoria_leaf_spot', 'Tomato_Spider_mites_Two_spotted_spider_mite', 'Tomato__Target_Spot', 'Tomato__Tomato_YellowLeaf__Curl_Virus', 'Tomato__Tomato_mosaic_virus', 'Tomato_healthy']


### 3. Model Architecture with Data Augmentation

In [4]:
inputs = layers.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3))

x = layers.RandomFlip("horizontal_and_vertical")(inputs)
x = layers.RandomRotation(0.15)(x)
x = layers.RandomZoom(0.1)(x)
x = layers.Rescaling(1. / 255)(x)

x = layers.Conv2D(32, (3, 3), padding="same", activation="relu")(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D((2, 2))(x)

x = layers.Conv2D(64, (3, 3), padding="same", activation="relu")(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D((2, 2))(x)

x = layers.Conv2D(128, (3, 3), padding="same", activation="relu")(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D((2, 2))(x)

x = layers.Conv2D(256, (3, 3), padding="same", activation="relu")(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D((2, 2))(x)

x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.4)(x)
outputs = layers.Dense(len(POTATO_TOMATO_CLASSES), activation="softmax")(x)

model = models.Model(inputs=inputs, outputs=outputs, name="IPDDS_CNN")
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
model.summary()

Model: "IPDDS_CNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_flip (RandomFlip)        │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_rotation                 │ (None, 128, 128, 3)    │             0 │
│ (RandomRotation)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_zoom (RandomZoom)        │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 128, 128, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128, 128, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 64, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64, 64, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 32, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 32, 32, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 16, 16, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 16, 16, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 8, 8, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 256)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 13)             │         1,677 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 424,909 (1.62 MB)

 Trainable params: 423,949 (1.62 MB)

 Non-trainable params: 960 (3.75 KB)

### 4. Training & Saving Model Weights

In [5]:
os.makedirs("../model", exist_ok=True)
metadata_key_by_class = {
    "Tomato_Bacterial_spot": "Tomato___Bacterial_spot",
    "Tomato_Early_blight": "Tomato___Early_blight",
    "Tomato_Late_blight": "Tomato___Late_blight",
    "Tomato_Leaf_Mold": "Tomato___Leaf_Mold",
    "Tomato_Septoria_leaf_spot": "Tomato___Septoria_leaf_spot",
    "Tomato_Spider_mites_Two_spotted_spider_mite": "Tomato___Spider_mites Two-spotted_spider_mite",
    "Tomato__Target_Spot": "Tomato___Target_Spot",
    "Tomato__Tomato_YellowLeaf__Curl_Virus": "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
    "Tomato__Tomato_mosaic_virus": "Tomato___Tomato_mosaic_virus",
    "Tomato_healthy": "Tomato___healthy"
}
class_metadata = {
    class_name: DISEASE_METADATA[metadata_key_by_class.get(class_name, class_name)]
    for class_name in POTATO_TOMATO_CLASSES
}
with open("../model/class_indices.json", "w", encoding="utf-8") as metadata_file:
    json.dump({
        "class_names": POTATO_TOMATO_CLASSES,
        "classes": class_metadata,
        "image_size": list(IMG_SIZE)
    }, metadata_file, indent=2)

checkpoint_cb = callbacks.ModelCheckpoint(
    "../model/disease_model_checkpoint.keras",
    save_best_only=True,
    monitor="val_accuracy",
    mode="max"
)
early_stop_cb = callbacks.EarlyStopping(
    monitor="val_accuracy",
    patience=4,
    restore_best_weights=True
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[checkpoint_cb, early_stop_cb]
)

model.save("../model/disease_model.keras")
model.save("../model/disease_model.h5")
print("Disease model and accurate class metadata saved to ../model/")

Epoch 1/10


C:\Users\Muneer Hameed\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


  1/364 ━━━━━━━━━━━━━━━━━━━━ 24:22 4s/step - accuracy: 0.1875 - loss: 3.0422

  2/364 ━━━━━━━━━━━━━━━━━━━━ 4:31 749ms/step - accuracy: 0.1719 - loss: 2.8743

  3/364 ━━━━━━━━━━━━━━━━━━━━ 4:32 755ms/step - accuracy: 0.1979 - loss: 2.7614

...

362/364 ━━━━━━━━━━━━━━━━━━━━ 1s 735ms/step - accuracy: 0.9088 - loss: 0.2706

363/364 ━━━━━━━━━━━━━━━━━━━━ 0s 735ms/step - accuracy: 0.9089 - loss: 0.2704

364/364 ━━━━━━━━━━━━━━━━━━━━ 0s 733ms/step - accuracy: 0.9090 - loss: 0.2704

364/364 ━━━━━━━━━━━━━━━━━━━━ 278s 764ms/step - accuracy: 0.9090 - loss: 0.2704 - val_accuracy: 0.7407 - val_loss: 0.9734


Disease model and accurate class metadata saved to ../model/


### 5. Train a Separate Crop-Type Classifier
A binary classifier is trained on the disease model's learned image features to distinguish Potato from Tomato independently of disease.

In [6]:
feature_extractor = models.Model(
    inputs=model.input,
    outputs=model.get_layer("dense").output,
    name="IPDDS_Feature_Extractor"
)
crop_labels_by_class = tf.constant(
    [0 if class_name.startswith("Potato") else 1 for class_name in POTATO_TOMATO_CLASSES],
    dtype=tf.int32
)

def extract_crop_features(dataset):
    feature_batches = []
    crop_label_batches = []
    for images, disease_labels in dataset:
        feature_batches.append(feature_extractor(images, training=False).numpy())
        crop_label_batches.append(tf.gather(crop_labels_by_class, disease_labels).numpy())
    return np.concatenate(feature_batches), np.concatenate(crop_label_batches)

train_features, train_crop_labels = extract_crop_features(train_ds)
val_features, val_crop_labels = extract_crop_features(val_ds)
crop_counts = np.bincount(train_crop_labels, minlength=2)
crop_class_weights = {
    index: len(train_crop_labels) / (2 * count)
    for index, count in enumerate(crop_counts)
}

crop_head = models.Sequential([
    layers.Input(shape=(feature_extractor.output_shape[-1],)),
    layers.Dense(2, activation="softmax")
], name="IPDDS_Crop_Head")
crop_head.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
crop_head.fit(
    train_features,
    train_crop_labels,
    validation_data=(val_features, val_crop_labels),
    class_weight=crop_class_weights,
    epochs=30,
    batch_size=128,
    callbacks=[callbacks.EarlyStopping(monitor="val_accuracy", patience=4, restore_best_weights=True)]
)

crop_model = models.Model(
    inputs=model.input,
    outputs=crop_head(feature_extractor.output),
    name="IPDDS_Crop_Classifier"
)
crop_model.save("../model/crop_type_model.keras")
print("Crop classifier saved to ../model/crop_type_model.keras (index 0: Potato, index 1: Tomato)")

Epoch 1/30


 1/91 ━━━━━━━━━━━━━━━━━━━━ 39s 437ms/step - accuracy: 0.8438 - loss: 2.6604

53/91 ━━━━━━━━━━━━━━━━━━━━ 0s 979us/step - accuracy: 0.7746 - loss: 0.9761 

91/91 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8061 - loss: 0.7116 - val_accuracy: 0.8864 - val_loss: 0.2578


Epoch 2/30


...

Epoch 22/30


 1/91 ━━━━━━━━━━━━━━━━━━━━ 2s 25ms/step - accuracy: 0.9531 - loss: 0.0643

56/91 ━━━━━━━━━━━━━━━━━━━━ 0s 916us/step - accuracy: 0.9640 - loss: 0.0819

91/91 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9631 - loss: 0.0841 - val_accuracy: 0.9680 - val_loss: 0.0819


Crop classifier saved to ../model/crop_type_model.keras (index 0: Potato, index 1: Tomato)
